In [10]:
# basic chat app

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI()

chat_history = [
    SystemMessage(content="You are a helpful assistant"), 

]

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break
    chat_history.append(HumanMessage(content=user_input))
    result = model.invoke(chat_history)
    chat_history.append(AIMessage(content=result.content))
    print(f"AI: {result.content}")


print(chat_history)


You:  hi


AI: Hello! How can I assist you today?


In [12]:
chat_history[0]

SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={})

In [13]:
chat_history[1]

HumanMessage(content='hi', additional_kwargs={}, response_metadata={})

In [14]:
chat_history[2]

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={})

In [19]:
chat_history

[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hi', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='tell me about llms', additional_kwargs={}, response_metadata={}),
 AIMessage(content='LLMS typically stands for Learning Management Systems. These are software applications used by organizations to manage, deliver, and track training and educational programs. LLMSs are commonly used in schools, universities, companies, and other institutions to enhance learning and development through online courses and virtual classrooms. They can help streamline the administrative aspects of learning, such as registration, course delivery, assessments, and tracking progress. Overall, LLMSs are helpful tools for creating and managing online learning experiences. Let me know if you have any specific questions a

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, messages_from_dict, messages_to_dict
from dotenv import load_dotenv
import json
import os

load_dotenv()

# File where chat history will be saved
HISTORY_FILE = "chat_history.json"

# 1️⃣ Load existing chat history if it exists
if os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, "r", encoding="utf-8") as f:
        chat_history = messages_from_dict(json.load(f))
    print("Loaded previous chat history.")
else:
    # Start with system message if no history exists
    chat_history = [SystemMessage(content="You are a helpful assistant")]

# Initialize the model
model = ChatOpenAI()

# 2️⃣ Chat loop
while True:
    user_input = input("You: ")
    
    # Exit condition
    if user_input.lower() == "exit":
        print("Exiting chat...")
        break
    
    # Append user's message
    chat_history.append(HumanMessage(content=user_input))
    
    # Get AI response
    result = model.invoke(chat_history)
    
    # Append AI message
    chat_history.append(AIMessage(content=result.content))
    
    # Print AI response
    print(f"AI: {result.content}")
    
    # 3️⃣ Save history after every message (optional but safe)
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(messages_to_dict(chat_history), f, ensure_ascii=False, indent=2)

# 4️⃣ Optional: print full chat history at the end
print("\nFull chat history:")
for msg in chat_history:
    role = "System" if isinstance(msg, SystemMessage) else "You" if isinstance(msg, HumanMessage) else "AI"
    print(f"{role}: {msg.content}")


Loaded previous chat history.


You:  hello


AI: Hi there! How can I help you today?


You:  exit


Exiting chat...

Full chat history:
System: You are a helpful assistant
You: hi
AI: Hello! How can I assist you today?
You: tell me about llms
AI: LLMS typically stands for Learning Management Systems. These are software applications used by organizations to manage, deliver, and track training and educational programs. LLMSs are commonly used in schools, universities, companies, and other institutions to enhance learning and development through online courses and virtual classrooms. They can help streamline the administrative aspects of learning, such as registration, course delivery, assessments, and tracking progress. Overall, LLMSs are helpful tools for creating and managing online learning experiences. Let me know if you have any specific questions about LLMS!
You: elaborate more
AI: Learning Management Systems (LMS) are software applications that provide a platform for delivering, managing, and tracking training and educational programs in a centralized location. Here are some key

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnableParallel, RunnablePassthrough
from dotenv import load_dotenv
load_dotenv()


# load PDF
file_path = "data/Student Handbook 2025_ENG.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load() 
transcript = " ".join([doc.page_content for doc in docs])

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks= splitter.create_documents([transcript])


# loading vector store
model = "OpenAI"
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
try:
    vector_store = FAISS.load_local("student_handbook_db", embedding, index_name=model, allow_dangerous_deserialization=True)
except Exception:
    vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)
    vector_store.save_local("student_handbook_db", index_name=model)



llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
parser = StrOutputParser()


retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={'k': 10, 'lambda_mult': 0.25})

aug_prompt = PromptTemplate(
                            template="""
                            You are a helpful assistant. 
                            Answer ONLY from the provided transcript context. 
                            Answer in bullet form.
                            If the context is insufficient, just say you don't know.
                            Be polite in your response.
                            Transcript: {context}
                            Question: {question}
                            """, 
                            input_variables=["context", "question"]
                            )

# chains
context_chain = RunnableSequence(retriever, RunnableLambda(lambda retrieved_docs: "\n\n".join([doc.page_content for doc in retrieved_docs])))
aug_prompt_chain = RunnableSequence(aug_prompt, llm)


parallel_chain = RunnableParallel({"context": context_chain, "question": RunnablePassthrough()})

chain = RunnableSequence(parallel_chain, aug_prompt_chain, parser)



    
user = "its a beautiful day today"

result = chain.invoke(user)
print(result)

I don't know.
